# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h0pi/flyrank-ml-intenrship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup: same slice, label and features as weeks 3–4

Lane: **Refresh / Content Opportunity Scoring**. Same data contract as `w03_data_contract.ipynb` and the same rows as the week-4 baseline (`w04_baseline_score.ipynb`):

- one row = one page with Google Search data in March 2026, decision moment **2026-03-16**;
- inputs only from **1–15 March** (plus creation date and content type);
- label `is_declining`: average daily impressions on 16–31 March < 80% of 1–15 March.

The first run reads the March partition from the release and caches a small per-page table in `work/outputs/` (git-ignored).

In [1]:

%pip -q install duckdb huggingface_hub scikit-learn

import os, json, getpass
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

SEED = 42
pd.set_option("display.width", 180)

def hf_token():
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not tok:
        try:
            from huggingface_hub import get_token
            tok = get_token()
        except Exception:
            pass
    return tok or getpass.getpass("Hugging Face READ token: ")

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "march_pages.parquet"          # same cache file as week 4

if not CACHE.exists():
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token()}')")
    REL = "hf://datasets/FlyRank/internship-warehouse"
    con.sql(f"""
        WITH d AS (
            SELECT client_hash_id, content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
            FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
            WHERE gsc_data_available IS TRUE
        ),
        p AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15')                            AS imp_h1,
                   SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15')                            AS clk_h1,
                   SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-07')                            AS imp_w1,
                   SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15') AS imp_w2,
                   SUM(gsc_avg_position * gsc_impressions)
                       FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0 AND gsc_avg_position > 0) AS pos_x_imp,
                   SUM(gsc_impressions)
                       FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0 AND gsc_avg_position > 0) AS imp_with_pos,
                   SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-16')                            AS imp_h2
            FROM d GROUP BY 1, 2
            HAVING SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') >= 100
        )
        SELECT p.*, c.content_created_date, c.content_type
        FROM p JOIN read_parquet('{REL}/dim_content.parquet') c USING (client_hash_id, content_hash_id)
        WHERE c.content_created_date < DATE '2026-03-01'
    """).df().to_parquet(CACHE)

df = pd.read_parquet(CACHE).sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
df["imp_h2"] = df["imp_h2"].fillna(0)
# ---- inputs: all knowable on 2026-03-16 ----
df["imp_per_day"]   = df["imp_h1"] / 15
df["log_imp"]       = np.log1p(df["imp_per_day"])
df["momentum"]      = np.log((df["imp_w2"].fillna(0) / 8 + 1) / (df["imp_w1"].fillna(0) / 7 + 1))
df["ctr"]           = df["clk_h1"] / df["imp_h1"]
df["avg_position"]  = df["pos_x_imp"] / df["imp_with_pos"]
df["age_days"]      = (pd.Timestamp("2026-03-15") - pd.to_datetime(df["content_created_date"])).dt.days
df["is_feedly"]     = (df["content_type"] == "feedly article").astype(int)
df["is_comparison"] = (df["content_type"] == "comparison article").astype(int)
# ---- label: 16-31 March, used only as the target ----
df["is_declining"]  = ((df["imp_h2"] / 16) < 0.8 * df["imp_per_day"]).astype(int)

FEATURES = ["log_imp", "momentum", "ctr", "avg_position", "age_days", "is_feedly", "is_comparison"]
assert df[FEATURES].notna().all().all()
# week-4 baseline rule, frozen exactly as it was
df["baseline_score"] = np.where(df["momentum"] < -0.1, -df["momentum"] * np.log1p(df["imp_per_day"]), 0.0)

print(f"pages: {len(df):,} | clients: {df.client_hash_id.nunique()} | base rate: {df.is_declining.mean():.3f}")
print(f"scikit-learn {sklearn.__version__} | seed {SEED}")


Note: you may need to restart the kernel to use updated packages.


pages: 74,377 | clients: 36 | base rate: 0.336
scikit-learn 1.9.1 | seed 42


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My question is a **ranking** one ("which pages should a content team review first?") on a **yes/no label I actually observed** (did the page decline in late March?). So the right tool is a classifier whose **probability** is used to rank pages, evaluated with **Precision@K**, the same metric as the week-4 baseline.

I train two models, simplest first:

1. **Logistic Regression** (features standardised). It is readable: one weight per feature, so I can check that the model leans on things that make sense. It is my main candidate.
2. **Histogram Gradient Boosting**. A stronger, non-linear model that can pick up interactions (e.g. "falling *and* low position"). It only gets picked if it clearly beats the simpler one at the K values that matter.

**Seven inputs**, all from 1–15 March or fixed before it: daily impressions (log), early momentum (week 2 vs week 1), CTR, impression-weighted position, page age, and content type (two flags). Momentum was the CONFIRMED signal in week 4. Age was MIXED, so it goes in as a plain input and the model decides how much it matters. No impressions from 16–31 March, no snapshot dates and no product flags.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**5-fold cross-validation grouped by client (`GroupKFold`).** Each fold's test set is a group of clients the model never saw in training. Why:

- **Pages of one client move together:** same site, same migrations, same tracking. A random split would put sister pages of the same client in both train and test, and the model would learn "this client is falling" instead of page-level signals. Week 4 already showed one client holding 6 of the top-20 pages.
- **It matches real use:** the score has to work for a client's pages it has not been tuned on.
- **Time-aware by construction:** every input comes from 1–15 March and the label from 16–31 March, so nothing from the future leaks into features.
- **Same split for the baseline:** the rule has no training, so I simply score it on the same test folds.

I report two views: the **per-fold average** (how well it works on a fresh set of clients) and a **pooled** ranking of all out-of-fold scores, which is comparable to the week-4 numbers.

Limitation, stated up front: all folds come from the same month, so this does not test whether the model still works in a different season.

In [2]:

gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(df, df["is_declining"], groups=df["client_hash_id"]))
df["fold"] = -1
for k, (tr, te) in enumerate(folds):
    df.loc[te, "fold"] = k
    assert set(df.client_hash_id.iloc[tr]).isdisjoint(df.client_hash_id.iloc[te]), "client leaked across the split"

(df.groupby("fold")
   .agg(pages=("is_declining", "size"), clients=("client_hash_id", "nunique"), base_rate=("is_declining", "mean"))
   .round(3))


,pages,clients,base_rate
fold,,,
0,17562,1,0.350
1,14151,9,0.431
2,14150,11,0.418
3,14153,8,0.233
4,14361,7,0.245


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:

def make_models():
    return {
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "Gradient Boosting": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED),
    }

oof = {name: np.zeros(len(df)) for name in make_models()}
fitted = {name: [] for name in make_models()}
for k, (tr, te) in enumerate(folds):
    for name, model in make_models().items():
        model.fit(df.loc[tr, FEATURES], df.loc[tr, "is_declining"])
        oof[name][te] = model.predict_proba(df.loc[te, FEATURES])[:, 1]
        fitted[name].append(model)
for name in oof:
    df[f"score_{name}"] = oof[name]

KS = [50, 100, 500, 1000]
SCORES = {"Baseline rule (week 4)": "baseline_score",
          "Logistic Regression": "score_Logistic Regression",
          "Gradient Boosting": "score_Gradient Boosting"}

def p_at_k(frame, col, k):
    return frame.sort_values(col, ascending=False)["is_declining"].head(k).mean()

rows = []
for label, col in SCORES.items():
    per_fold = [{**{f"P@{k}": p_at_k(df[df.fold == f], col, k) for k in KS},
                 "ROC-AUC": roc_auc_score(df.loc[df.fold == f, "is_declining"], df.loc[df.fold == f, col])} for f in range(5)]
    pf = pd.DataFrame(per_fold)
    rows.append({"method": label, **{f"{m} (fold mean)": pf[m].mean() for m in pf.columns},
                 "ROC-AUC (fold sd)": pf["ROC-AUC"].std(),
                 **{f"P@{k} (pooled)": p_at_k(df, col, k) for k in KS}})
table = pd.DataFrame(rows).set_index("method")
table.loc["Base rate (random pick)"] = df["is_declining"].mean()
table.loc["Base rate (random pick)", "ROC-AUC (fold mean)"] = 0.5
table.loc["Base rate (random pick)", "ROC-AUC (fold sd)"] = np.nan
table.round(3).T


method,Baseline rule (week 4),Logistic Regression,Gradient Boosting,Base rate (random pick)
P@50 (fold mean),0.864,0.912,0.892,0.336
P@100 (fold mean),0.816,0.894,0.868,0.336
P@500 (fold mean),0.694,0.754,0.745,0.336
P@1000 (fold mean),0.613,0.687,0.681,0.336
ROC-AUC (fold mean),0.624,0.666,0.679,0.500
ROC-AUC (fold sd),0.052,0.053,0.040,NaN
P@50 (pooled),0.980,0.980,0.880,0.336
P@100 (pooled),0.960,0.970,0.880,0.336
P@500 (pooled),0.832,0.880,0.856,0.336
P@1000 (pooled),0.779,0.825,0.809,0.336


In [4]:

metrics = {"split": "GroupKFold(5) by client_hash_id", "seed": SEED, "sklearn": sklearn.__version__,
           "rows": int(len(df)), "base_rate": round(float(df.is_declining.mean()), 4),
           "results": {m: {k: (None if pd.isna(v) else round(float(v), 4)) for k, v in r.items()}
                       for m, r in table.round(4).iterrows()}}
(OUT / "w05_model_metrics.json").write_text(json.dumps(metrics, indent=2))
print("wrote work/outputs/w05_model_metrics.json")


wrote work/outputs/w05_model_metrics.json


**Reading the table** (fold mean = average over 5 groups of unseen clients; pooled = one ranking over all out-of-fold scores):

- **Logistic Regression beats the baseline rule where it matters for a review queue longer than a handful of pages.** On the per-fold averages it is higher at every K, and in the pooled ranking it reaches P@500 ≈ 0.88 vs 0.83 and P@1000 ≈ 0.83 vs 0.78. Its ROC-AUC is also higher (≈ 0.67 vs 0.62 per fold).
- **At the very top they tie:** pooled P@50 is 0.98 for both. The rule's top picks are the "cliff drops" from week 4, which almost always keep falling, so there is little room left to win there.
- **Gradient Boosting has the best ROC-AUC (≈ 0.68) but is worse at the top of the list** (pooled P@50 = 0.88). It orders the middle of the list better, but the content team reads the top. **More complexity did not earn its place, so I pick Logistic Regression.**
- **The spread across folds is large** (ROC-AUC sd ≈ 0.05). Fold 0 is a single large client with 17,562 pages, and the folds' base rates range from 23% to 43%, so differences of one or two points are within noise. The LR-vs-baseline gap at P@500 and P@1000 holds in the per-fold view too.

Week 3's quick Random Forest scored 0.721 ROC-AUC on a single client split. With 5 grouped folds the honest average is lower: that one split was a lucky set of test clients.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### What the model leans on

In [5]:

# Logistic Regression weights (standardised inputs, averaged over the 5 folds)
coefs = pd.DataFrame([m[-1].coef_[0] for m in fitted["Logistic Regression"]], columns=FEATURES)
weights = pd.DataFrame({"mean weight": coefs.mean(), "sd over folds": coefs.std()}).sort_values("mean weight")

# permutation importance on each held-out fold: drop in ROC-AUC when one input is shuffled
imp = []
for k, (tr, te) in enumerate(folds):
    r = permutation_importance(fitted["Logistic Regression"][k], df.loc[te, FEATURES], df.loc[te, "is_declining"],
                               scoring="roc_auc", n_repeats=5, random_state=SEED)
    imp.append(r.importances_mean)
weights["AUC drop when shuffled"] = pd.Series(np.mean(imp, axis=0), index=FEATURES)
weights.round(3)


,mean weight,sd over folds,AUC drop when shuffled
momentum,-0.752,0.051,0.121
ctr,-0.323,0.032,0.028
age_days,-0.279,0.045,0.021
log_imp,0.001,0.073,-0.004
is_comparison,0.036,0.021,0.000
is_feedly,0.049,0.015,0.001
avg_position,0.146,0.098,0.005


- **Momentum is by far the most important input:** shuffling it costs the most ROC-AUC, and its weight is strongly negative (a page that was already sliding is more likely to keep sliding). That is the CONFIRMED signal from week 4, now learned from data rather than hand-coded.
- **CTR and page age come next, both with negative weights.** Pages that already get a higher click-through rate decline less, and **older pages decline less**. That is the opposite of the "stale pages decay" assumption behind the refresh flags, and it matches the week-4 staleness check.
- **Volume (`log_imp`) has essentially zero weight**, consistent with week 4's flat decline rate across volume quartiles. Position and content type add almost nothing.
- No input is "suspiciously perfect" (the top one costs 0.12 AUC when shuffled, not 0.4), which is a good sign that no leakage got in.

### Where the model is wrong

In [6]:

col = "score_Logistic Regression"
df["momentum_bucket"] = pd.cut(df["momentum"], [-99, -0.3, -0.1, 0.1, 99],
                               labels=["falling hard", "falling", "flat", "rising"])

def group_table(by):
    rows = []
    for g, part in df.groupby(by, observed=True):
        if part["is_declining"].nunique() < 2:
            continue
        rows.append({by: g, "pages": len(part), "base_rate": part["is_declining"].mean(),
                     "ROC-AUC model": roc_auc_score(part["is_declining"], part[col]),
                     "ROC-AUC baseline": roc_auc_score(part["is_declining"], part["baseline_score"])})
    return pd.DataFrame(rows).set_index(by).round(3)

display(group_table("content_type"))
display(group_table("momentum_bucket"))


,pages,base_rate,ROC-AUC model,ROC-AUC baseline
content_type,,,,
comparison article,523,0.612,0.517,0.511
feedly article,546,0.452,0.634,0.662
keyword article,73308,0.333,0.657,0.633


,pages,base_rate,ROC-AUC model,ROC-AUC baseline
momentum_bucket,,,,
falling hard,18531,0.537,0.645,0.581
falling,14482,0.323,0.587,0.492
flat,15672,0.287,0.549,0.500
rising,25692,0.228,0.562,0.500


- **By content type:** the model beats the rule on `keyword article` pages (98% of the data), is about equal on `comparison` and **slightly worse than the rule on `feedly` articles** (0.63 vs 0.66). Those small groups give it little to learn from, so for news-style pages the simple rule is just as good.
- **By momentum:** inside each momentum bucket the model's ROC-AUC drops to 0.55–0.65. Most of its power comes from *separating* sliding pages from rising ones. It does add something the rule cannot: among flat and rising pages, where the rule gives everyone the same score (0.50), the model still ranks a little better than chance. Telling which of the "falling hard" pages will keep falling is the hardest part, and better inputs (query mix, longer history) would be needed there.

### Three concrete wrong cases

In [7]:

show = ["client_hash_id", "content_type", "imp_per_day", "momentum", "avg_position", "age_days", col, "is_declining"]
fp = df[df.is_declining == 0].nlargest(2, col)      # confident "will decline", but it didn't
fn = df[df.is_declining == 1].nsmallest(1, col)     # confident "safe", but it declined
cases = pd.concat([fp, fn])[show].copy()
cases["client_hash_id"] = cases["client_hash_id"].str[:14] + "…"
cases["imp/day 16–31 Mar"] = (df.loc[cases.index, "imp_h2"] / 16).round(1)
cases.round(3)


,client_hash_id,content_type,imp_per_day,momentum,avg_position,age_days,score_Logistic Regression,is_declining,imp/day 16–31 Mar
8870,client_2094c6e…,keyword article,8.200,-2.683,48.374,26,0.970,0,25.0
66719,client_e5c2aa2…,keyword article,20.133,-2.832,3.253,68,0.955,0,30.2
52772,client_73cda7b…,keyword article,358.067,6.511,6.353,17,0.000,1,228.8


- **Cases 1–2 (false alarms):** the model was very sure these pages would decline, because their impressions dropped sharply from week 1 to week 2, but in late March they recovered (to ~25 and ~30/day). A one-week dip (a holiday, a reporting glitch, a temporary ranking swing) looks exactly like the start of a real decline in 15 days of data. Hard, because the inputs genuinely cannot tell them apart.
- **Case 3 (missed decline):** a brand-new page (17 days old) that was surging in early March (momentum +6.5). The model reads strong growth as "safe", but the launch surge then settled from ~358 to ~229 impressions/day, which counts as a decline under my label. For very new pages a settle-down after launch is normal, so this one is partly a label question: the week-4 review already showed new pages behave differently.

**Bottom line (decision support, not a guarantee):** a simple, readable Logistic Regression ranks pages for review measurably better than the week-4 rule on unseen clients (higher at every K per fold, and beyond the first ~100 picks in the pooled ranking), ties it at the very top of the pooled list, and leans on the same signal the rule did. The remaining errors are mostly short dips that recover and new pages settling after launch: patterns two weeks of data cannot separate from real decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.